# Introduction to Knowledge Graphs with Neo4j

This notebook demonstrates the basics of building and querying a knowledge graph using Neo4j.

## 1. Import Dependencies

In [ ]:
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase

## 2. Load Environment Variables

In [ ]:
load_dotenv()

AURA_INSTANCENAME = os.environ["AURA_INSTANCENAME"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

## 3. Initialize Neo4j Driver

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH, database=NEO4J_DATABASE)

## 4. Connect and Query Function

In [ ]:
def connect_and_query():
    """Connect to Neo4j and count the number of nodes."""
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            result = session.run("MATCH (n) RETURN count(n)")
            count = result.single().value()
            print(f"Number of nodes: {count}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()

## 5. Create Entities Function

In [ ]:
def create_entities(tx):
    """Create nodes for Albert Einstein and related entities."""
    # Create Albert Einstein node
    tx.run("MERGE (a:Person {name: 'Albert Einstein'})")

    # Create other nodes
    tx.run("MERGE (p:Subject {name: 'Physics'})")
    tx.run("MERGE (n:NobelPrize {name: 'Nobel Prize in Physics'})")
    tx.run("MERGE (g:Country {name: 'Germany'})")
    tx.run("MERGE (u:Country {name: 'USA'})")

## 6. Create Relationships Function

In [ ]:
def create_relationships(tx):
    """Create relationships between Albert Einstein and other entities."""
    # Create studied relationship
    tx.run(
        """
    MATCH (a:Person {name: 'Albert Einstein'}), (p:Subject {name: 'Physics'})
    MERGE (a)-[:STUDIED]->(p)
    """
    )

    # Create won relationship
    tx.run(
        """
    MATCH (a:Person {name: 'Albert Einstein'}), (n:NobelPrize {name: 'Nobel Prize in Physics'})
    MERGE (a)-[:WON]->(n)
    """
    )

    # Create born in relationship
    tx.run(
        """
    MATCH (a:Person {name: 'Albert Einstein'}), (g:Country {name: 'Germany'})
    MERGE (a)-[:BORN_IN]->(g)
    """
    )

    # Create died in relationship
    tx.run(
        """
    MATCH (a:Person {name: 'Albert Einstein'}), (u:Country {name: 'USA'})
    MERGE (a)-[:DIED_IN]->(u)
    """
    )

## 7. Simple Query Function

In [ ]:
def query_graph_simple(cypher_query):
    """Execute a simple Cypher query and print node names."""
    driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            result = session.run(cypher_query)
            for record in result:
                print(record["name"])
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()

## 8. Query Graph Function (Returns Paths)

In [ ]:
def query_graph(cypher_query):
    """Execute a Cypher query and print paths."""
    driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            result = session.run(cypher_query)
            for record in result:
                print(record["path"])
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()

## 9. Build Knowledge Graph Function

In [ ]:
def build_knowledge_graph():
    """Build the knowledge graph by creating entities and relationships."""
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            # Create entities
            session.execute_write(create_entities)
            # Create relationships
            session.execute_write(create_relationships)
            print("Knowledge graph built successfully!")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()

## 10. Define Cypher Queries

In [ ]:
# Cypher query to find paths related to Albert Einstein
einstein_query = """
MATCH path=(a:Person {name: 'Albert Einstein'})-[:STUDIED]->(s:Subject)
RETURN path
UNION
MATCH path=(a:Person {name: 'Albert Einstein'})-[:WON]->(n:NobelPrize)
RETURN path
UNION
MATCH path=(a:Person {name: 'Albert Einstein'})-[:BORN_IN]->(g:Country)
RETURN path
UNION
MATCH path=(a:Person {name: 'Albert Einstein'})-[:DIED_IN]->(u:Country)
RETURN path
"""

# Simple Cypher query to find all node names
simple_query = """
MATCH (n)
RETURN n.name AS name
"""

## 11. Execute - Build Knowledge Graph

Uncomment the line below to build the knowledge graph (run only once).

In [ ]:
# build_knowledge_graph()

## 12. Execute - Query All Node Names

Uncomment the line below to list all node names.

In [ ]:
# query_graph_simple(simple_query)

## 13. Execute - Query Einstein Relationships

In [ ]:
query_graph(einstein_query)

## Note: View the Entire Graph in Neo4j Browser

Run this query in the Neo4j browser/console to visualize the entire graph:

```cypher
MATCH (n)-[r]->(m)
RETURN n, r, m;
```